# Negflo with missing data

Subtracting routed upstream flow from downstream gauged flow gives a **residual** inflow, which can go negative
because the routing never matches the real system exactly. Clipping the negatives to zero inflates the residual
volume, so `Negflo` offers several ways of **redistributing** the negatives by reducing positive flows instead.

This notebook runs every method on one short example containing missing data, and shows all the results side by side
in a single table against the raw residual.

A few terms used throughout:

* **Flow event**: a run of consecutive days with flow above the flow limit (zero unless stated otherwise).
* **Absorbing**: an event takes on negative volume by scaling down each flow's excess above the flow limit, in
  proportion. Flows never go below the flow limit. If the event is too small, its flows are set to the flow limit and
  the rest of the negative volume is left over.
* **Unallocated negatives**: negative volume that no event absorbed, reported in `unallocated_negatives`. For every
  method, `total of result = total of residual - unallocated negatives`.

**Missing data** (NaN) contributes nothing to any total, ends a flow event, stays missing in the result, and
negative volume carries across it as if it were a day of zero flow.

In [1]:
import logging

import numpy as np
import pandas as pd

import bulum
from bulum.stats import Negflo

logging.getLogger("bulum").setLevel(logging.WARNING)  # hide info messages about unallocated negatives
print('bulum version: ', bulum.__version__)

bulum version:  0.3.4


## Example data

Measured (downstream) and modelled (routed upstream) flows over 45 days. The measured record has three gaps. In one of
them the gauge failed partway through an event, which splits that event in two.
`Negflo.from_flows` computes the residual as measured - modelled, and marks a day missing if either input is missing.

In [2]:
nan = np.nan
dates = pd.date_range("2000-01-01", periods=45)
residual = np.array([
    -4, -6, -3, -1,                 # negatives before the first event
    5, 18, 30, 22, 12, 6,           # event A
    -5, -8, -4,                     # negatives ...
    nan, nan, nan, nan,             # ... a gap ...
    -3, -2, 0,                      # ... and more negatives
    10, 35, 60,                     # event B, interrupted by a gap
    nan, nan,
    35, 20, 9,                      # event C (the rest of the interrupted event)
    -6, -10, -7, -2,
    4, 7, 3, 1,                     # event D (small)
    -1, -2, 0,
    nan, nan, nan,
    6, 3,                           # event E (small)
    -2,                             # negatives after the last event
])
modelled = pd.DataFrame({"flow": np.round(20 + 8 * np.sin(np.arange(45) * 2 * np.pi / 30), 1)}, index=dates)
measured = modelled + residual[:, None]   # NaN on the missing days

negflo = Negflo.from_flows(measured, modelled)
raw = negflo.rw1()["flow"]
print(f"Total positive residual: {raw[raw > 0].sum():.0f} ML")
print(f"Total negative residual: {raw[raw < 0].sum():.0f} ML")
print(f"Total residual:          {raw.sum():.0f} ML")
print(f"Missing days:            {raw.isna().sum()}")

Total positive residual: 286 ML
Total negative residual: -66 ML
Total residual:          220 ML
Missing days:            9


## All methods against the raw residual

Each column is the result of one method, alongside the raw residual (RW1). The **Event** column labels the flow events
A to E referred to in the explanations below. Missing days are greyed out.

The last two columns are variations on SM2, explained at the end: one with a flow limit of 5 ML/d, and one that treats
missing data as zero flow, as the original NEGFLO6 program did.

The bottom rows give each column's total, its unallocated negatives, and the residual total minus the unallocated
negatives. That last row always equals the total.

In [3]:
segments = [(dates[0], dates[19]), (dates[20], dates[-1])]   # used by SM6
runs = [
    ("RW1 (raw)", Negflo.from_flows(measured, modelled), "rw1"),
    ("CL1", Negflo.from_flows(measured, modelled), "cl1"),
    ("SM1", Negflo.from_flows(measured, modelled), "sm1"),
    ("SM2", Negflo.from_flows(measured, modelled), "sm2"),
    ("SM3", Negflo.from_flows(measured, modelled), "sm3"),
    ("SM4", Negflo.from_flows(measured, modelled), "sm4"),
    ("SM5", Negflo.from_flows(measured, modelled), "sm5"),
    ("SM6", Negflo.from_flows(measured, modelled, segments=segments), "sm6"),
    ("SM7", Negflo.from_flows(measured, modelled), "sm7"),
    ("SM2, flow limit 5", Negflo.from_flows(measured, modelled, flow_limit=5), "sm2"),
    ("SM2, gaps as zero", Negflo.from_flows(measured.fillna(0), modelled), "sm2"),
]
results, unallocated, residual_totals = {}, {}, {}
for name, nf, method in runs:
    results[name] = getattr(nf, method)()["flow"]
    unallocated[name] = nf.unallocated_negatives["flow"]
    residual_totals[name] = nf.rw1()["flow"].sum()
table = pd.DataFrame(results)

# label the flow events (runs of positive residual)
is_event = (raw > 0).astype(int)
event_number = (is_event.diff().fillna(is_event) == 1).cumsum()
table.insert(0, "Event", np.where(is_event == 1, event_number.map(lambda n: "ABCDEFGH"[n - 1]), ""))
table.index = table.index.strftime("%d %b")

summary = pd.DataFrame([table.drop(columns="Event").sum(), pd.Series(unallocated)],
                       index=["Total", "Unallocated negatives"])
summary.loc["Residual total - unallocated"] = pd.Series(residual_totals) - summary.loc["Unallocated negatives"]
summary.insert(0, "Event", "")
full = pd.concat([table, summary])

missing_rows = list(table.index[raw.isna().to_numpy()])
summary_rows = list(summary.index)
(full.style
     .format("{:.1f}", subset=full.columns[1:], na_rep="missing")
     .set_properties(subset=(missing_rows, full.columns), **{"background-color": "#ebeae6", "color": "#8a8984"})
     .set_properties(subset=(summary_rows, full.columns), **{"font-weight": "bold", "border-top": "1px solid #c8c7c2"})
     .set_properties(subset=(full.index, ["RW1 (raw)"]), **{"border-right": "2px solid #c8c7c2"})
     .set_table_styles([{"selector": "th", "props": "text-align: right; padding: 2px 8px;"},
                        {"selector": "th.col_heading", "props": "max-width: 6em; vertical-align: bottom;"},
                        {"selector": "td", "props": "text-align: right; padding: 2px 8px;"}]))

,Event,RW1 (raw),CL1,SM1,SM2,SM3,SM4,SM5,SM6,SM7,"SM2, flow limit 5","SM2, gaps as zero"
01 Jan,,-4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
02 Jan,,-6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
03 Jan,,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
04 Jan,,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
05 Jan,A,5.0,5.0,3.8,4.2,4.2,3.8,3.8,3.1,4.2,5.0,4.2
06 Jan,A,18.0,18.0,13.8,15.3,15.3,13.7,13.7,11.0,15.3,15.1,15.3
07 Jan,A,30.0,30.0,23.1,25.5,25.5,22.9,22.9,18.4,25.5,24.4,25.5
08 Jan,A,22.0,22.0,16.9,18.7,18.7,16.8,16.8,13.5,18.7,18.2,18.7
09 Jan,A,12.0,12.0,9.2,10.2,10.2,9.2,9.2,7.4,10.2,10.4,10.2
10 Jan,A,6.0,6.0,4.6,5.1,5.1,4.6,4.6,3.7,5.1,5.8,5.1


## What each method did

The event sizes in the raw residual are A 93 ML, B 105 ML, C 64 ML, D 15 ML and E 9 ML, and there are 66 ML of
negatives in total.

**CL1** simply sets the negatives to zero. That's easy, but the total is now the **positive** residual, so it overstates
the residual volume by the full 66 ML.

**SM1** scales all positive flows by the same factor, `1 - total negative / total positive`, so every event shrinks by
the same proportion and the total matches the residual (no unallocated negatives). The missing days take no part in either total.
SM1 ignores the flow limit.

**SM2** (into the following event, with carry-over): each event absorbs the negatives **before** it.

* Event A absorbs the 14 ML of negatives at the start.
* The 22 ML of negatives after event A (on both sides of the first gap) **carry across the gap** and are absorbed by
  event B.
* The gap during the large event splits it into events B and C. Nothing negative lies between them, so event C is
  untouched.
* The 25 ML before event D is more than D's 15 ML, so D is reduced to zero and the remaining 10 ML carries over to
  event E, which is also reduced to zero.
* What event E couldn't absorb, plus the 2 ML at the end with no following event, is left unallocated.

**SM3** is SM2 without carry-over. Event D is still reduced to zero, but its 10 ML shortfall is discarded, so event E
only absorbs the 3 ML just before it and survives. The discarded volume shows up as larger unallocated negatives.

**SM4** (into the preceding event, with carry-over): each event absorbs the negatives **after** it.

* The 14 ML at the start has no preceding event, so (as in NEGFLO6) it is not redistributed: it is left unallocated,
  and the total is 14 ML more than the residual.
* Event A absorbs the 22 ML after it (again across the gap).
* Event B is followed only by a gap, so it's untouched.
* Event C absorbs the 25 ML after it, D the 3 ML after it, and E the final 2 ML.

**SM5** is SM4 without carry-over. Here every event is large enough to absorb the negatives after it, so nothing needs
carrying and SM5 gives the same result as SM4.

**SM6** applies SM1 separately within each segment, here 1-20 Jan and 21 Jan onwards. The first segment has a higher
ratio of negative to positive flow, so event A is reduced proportionally more than the later events.

**SM7** gives each block of negatives to whichever of the events either side of it is larger. The 22 ML between
A (93 ML) and B (105 ML) goes to B, and the 25 ML between C and D goes to C. Negatives at the start and end go to the
only neighbouring event, and no volume is carried over.

**SM2, flow limit 5**: events are now runs of flow **above** 5 ML/d, and only the excess above 5 is reduced. The small
flows (e.g. the 4, 3 and 1 in event D) are kept, and no flow is reduced below 5 ML/d. The cost is that less volume is
available to absorb the negatives, so more are left unallocated.

**SM2, gaps as zero**: the original NEGFLO6 program read a missing measured flow as zero, which turns every missing day
into a negative residual equal to the modelled flow. Those spurious negatives (zero in this column on the missing days)
wipe out event B and cut event C, and the total falls far below the real residual. Excluding missing data avoids this.

In [4]:
print(f"Spurious negatives NEGFLO6 would create from the missing days: {modelled['flow'][raw.isna()].sum():.0f} ML")

Spurious negatives NEGFLO6 would create from the missing days: 188 ML
